# 09 — Renewables insights (Genie)

Provisions **Energy Trading Genie - Renewables** over wind, solar, published net volume, and accuracy tables.

**Depends on:** volume-forecasting notebooks `04`–`08`.

**Produces:** Genie Space for L1 demo — *"Show me renewables book performance…"*

In [ ]:
# Databricks notebook source
import os
from pathlib import Path

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))
dbutils.widgets.text("warehouse_id", "")

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
WAREHOUSE_WIDGET = dbutils.widgets.get("warehouse_id").strip()

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")

def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

print(f"Target: {CATALOG}.{SCHEMA}")

In [ ]:
REQUIRED = [
    "volume_forecast_gold_wind_summary",
    "volume_forecast_gold_solar_summary",
    "volume_forecast_gold_net_volume",
    "volume_forecast_gold_accuracy_daily",
]
missing = [t for t in REQUIRED if not spark.catalog.tableExists(fq(t))]
assert not missing, f"Run vf04–vf08 first; missing: {missing}"

In [ ]:
_cfg_paths = [Path.cwd() / "genie_space_config.py"]
try:
    _nb = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    )
    _cfg_paths.insert(0, Path(_nb).parent / "genie_space_config.py")
except Exception:
    pass
_cfg_py = next((p for p in _cfg_paths if p.is_file()), None)
if _cfg_py is None:
    raise FileNotFoundError("genie_space_config.py not found next to this notebook")
_fq_table = fq
_cfg_globals = {**globals(), "__file__": str(_cfg_py)}
exec(_cfg_py.read_text(), _cfg_globals)
for _k in _cfg_globals:
    if _k not in ("_cfg_globals", "_fq_table", "_cfg_py", "_cfg_paths", "_nb"):
        globals()[_k] = _cfg_globals[_k]
fq = _fq_table
print("Loaded", _cfg_py.name)

In [ ]:
created_mvs = []
mv_errors = []
for spec in METRIC_VIEW_SPECS:
    sql = build_metric_view_sql(CATALOG, SCHEMA, spec)
    try:
        spark.sql(sql)
        created_mvs.append(spec[0])
    except Exception as exc:
        mv_errors.append((spec[0], str(exc)))

print("Metric views created:", created_mvs)
if mv_errors:
    for name, err in mv_errors:
        print(f"  {name}: {err[:200]}")

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import State

w = WorkspaceClient()

def _pick_warehouse() -> str:
    if WAREHOUSE_WIDGET:
        return WAREHOUSE_WIDGET
    running, other = [], []
    for wh in w.warehouses.list():
        wid = (wh.id or "").strip()
        if not wid:
            continue
        (running if wh.state == State.RUNNING else other).append(wid)
    if running:
        return running[0]
    if other:
        return other[0]
    raise RuntimeError("No SQL warehouse — set warehouse_id widget")

warehouse_id = _pick_warehouse()
include_mvs = len(created_mvs) == len(METRIC_VIEW_SPECS)
serialized = build_serialized_space(CATALOG, SCHEMA, include_metric_views=include_mvs)

space, operation = provision_genie_space(
    w,
    title=GENIE_SPACE_TITLE,
    description=GENIE_SPACE_DESCRIPTION,
    warehouse_id=warehouse_id,
    serialized_space=serialized,
)

print(f"Genie Space {operation}: {GENIE_SPACE_TITLE}")
print(f"  space_id:     {space.space_id}")
print(f"  warehouse_id: {warehouse_id}")